In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import re
import seaborn as sns

from scipy.stats import linregress

sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
with open("../../data/accession_translator.json", "r") as s:
    accession_translator = json.load(s)
with open("../../data/tool_translator.json", "r") as s:
    datasets = json.load(s)

In [ ]:
with open("../../data/benchmarking_summary.json") as f:
    data = json.load(f)

size_mapping = {}
for entry in data["reports"]:
    accession = entry["accession"]
    size = entry["assembly_stats"]["total_sequence_length"]
    size_mapping[accession] = int(size)

In [ ]:
image_to_tool = {
    "annevo" : "ANNEVO",
    "augustus" : "AUGUSTUS",
    "teambraker/braker3:v3.0.7.6" : "BRAKER3",
    "helixer-with-models" : "Helixer",
    "geneml" : "geneML",
}

In [ ]:
def extract_docker_image(command):
    pattern = r'docker run\s+(?:--rm\s+|--cpuset-cpus=\S+\s+|--cpuset-mems=\S+\s+|-v\s+\S+\s+|--shm-size=\S+\s+)*(\S+)'
    match = re.search(pattern, command)
    return match.group(1) if match else None

In [ ]:
def extract_accession(text):
    matches = re.findall(r'GC[AF]_\d+\.\d+', text)
    if not matches:
        raise ValueError("No accession found")
    if len(set(matches)) > 1:
        raise ValueError(f"Multiple different accessions found: {set(matches)}")
    return matches.pop()

In [ ]:
joblog = pd.read_csv("../../intermediates/benchmarking_log.txt", delimiter='\t')
image = joblog["Command"].apply(extract_docker_image)
tool = image.map(image_to_tool)
accession = joblog["Command"].apply(extract_accession)
species = accession.map(accession_translator)
time_hours = joblog["JobRuntime"] / 3600
time_minutes = joblog["JobRuntime"] / 60
timings = pd.DataFrame({
    "accession": accession,
    "species": species,
    "tool": tool,
    "runtime_hours": time_hours,
    "runtime_minutes": time_minutes,
})
timings.to_csv("../../tables/Table_S7.tsv", sep='\t', index=False)

In [ ]:
runtime_summary = (
    timings.groupby("tool", observed=False)
    .agg(
        runtime_minutes_mean=("runtime_minutes", "mean"),
        runtime_minutes_std=("runtime_minutes", "std"),
        runtime_hours_mean=("runtime_hours", "mean"),
        runtime_hours_std=("runtime_hours", "std"),
    )
    .reset_index()
)

runtime_summary["tool"] = pd.Categorical(runtime_summary["tool"], categories=datasets.values(), ordered=True)
runtime_summary = runtime_summary.sort_values("tool").reset_index(drop=True)

runtime_summary_table = runtime_summary.assign(
    **{
        "runtime_minutes": runtime_summary.apply(
            lambda row: f"{row['runtime_minutes_mean']:.2f} ± {row['runtime_minutes_std']:.2f}", axis=1
        ),
        "runtime_hours": runtime_summary.apply(
            lambda row: f"{row['runtime_hours_mean']:.2f} ± {row['runtime_hours_std']:.2f}", axis=1
        ),
    }
)[["tool", "runtime_minutes", "runtime_hours"]]

runtime_summary_table.to_csv("../../tables/Table_3.tsv", sep='\t', index=False)

In [ ]:
timings["genome_size"] = timings["accession"].map(size_mapping)
timings["genome_size"] = timings["genome_size"] / 1e6

In [ ]:
tool_palette = {
    "ANNEVO": "#009E73",
    "AUGUSTUS": "#0072B2",
    "BRAKER3": "#FF81C6",
    "Helixer": "#462288",
    "geneML": "#FF822F",
}

In [ ]:
ax = sns.scatterplot(
    data=timings,
    x="genome_size",
    y="runtime_hours",
    hue="tool",
    palette=tool_palette
)

# Store slopes for each tool
slopes = {}

# Fit log–log regression per tool
for tool, group in timings.groupby("tool"):
    x = group["genome_size"]
    y = group["runtime_hours"]

    # Remove non-positive values (log undefined for <= 0)
    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]

    # Fit regression in log–log space
    slope, intercept, r, p, stderr = linregress(np.log10(x), np.log10(y))
    slopes[tool] = slope

    # Generate predicted line
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = 10**(intercept + slope * np.log10(x_line))

    # Plot regression line (exclude from legend)
    ax.plot(x_line, y_line, color=tool_palette[tool], lw=2, label="_nolegend_")

    # Add slope annotation at the end of each regression line
    ax.text(x_line[-1], y_line[-1], f"  {slope:.2f}",
            color=tool_palette[tool], fontsize=9, va="center", fontweight="bold")

# Log–log axes
ax.set_xscale("log")
ax.set_yscale("log")

# Axis titles
ax.set_xlabel("Genome size (Mb)")
ax.set_ylabel("Runtime (hours)")

# Axis limits and ticks
ax.set_xlim(10, 200)
ax.set_ylim(0.01, 60)
ax.xaxis.set_major_locator(mtick.FixedLocator([10, 20, 50, 100, 200]))
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(mtick.NullFormatter())
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}"))
ax.yaxis.set_minor_formatter(mtick.NullFormatter())

# Show x-axis grid lines at labeled ticks
ax.grid(True, axis="x", which="major")

# Place legend on the right side
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0)
plt.savefig("../../figures/Figure_S3.svg", bbox_inches="tight")

In [ ]:
# Log-log regression summary per tool: runtime = a * (genome_size ** b)
loglog_rows = []

for tool, group in timings.groupby("tool"):
    x = group["genome_size"]
    y = group["runtime_hours"]

    mask = (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]

    slope, intercept, r, p, stderr = linregress(np.log10(x), np.log10(y))
    a = 10 ** intercept
    b = slope
    r2 = r ** 2

    loglog_rows.append({
        "Tool": tool,
        "a": a,
        "b": b,
        "r": r,
        "R2": r2,
        "p_value": p,
        "n": len(x),
    })

loglog_df = pd.DataFrame(loglog_rows).sort_values("b")
loglog_df.to_csv("../../tables/Table_S8.tsv", sep='\t', index=False)